# Complexity of Statistics

## Datasaurus Dozen

**All Code in this notebook was generated by ChatGPT using the following two prompts**:
1. write the code for a jupyter notebook to display and animate the datasaurus dozen thirteen data sets
2. add the mean of x, mean of y, sample variance of x, sample variance of y, the correlation between x and y, and the linear regression line equation, and the r-squared value on the animation


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Load Datasaurus Dozen
url = (
    "https://raw.githubusercontent.com/rfordatascience/tidytuesday/"
    "main/data/2020/2020-10-13/datasaurus.csv"
)

df = pd.read_csv(url)

df.head()

In [ ]:
datasets = df['dataset'].unique()

fig, ax = plt.subplots(4, 4, figsize=(10, 10))

for a, dataset in zip(ax.flat, datasets):

    data = df[df['dataset'] == dataset]

    a.scatter(data.x, data.y, s=12)

    a.set_xlim(0, 100)
    a.set_ylim(0, 100)

    a.set_title(dataset)

# Remove unused axes
for a in ax.flat[len(datasets):]:
    a.axis('off')

plt.tight_layout()

In [ ]:
datasets = df['dataset'].unique()

# Put each dataset into an array of x,y coordinates
points = []

for dataset in datasets:
    data = df[df['dataset'] == dataset]
    points.append(data[['x', 'y']].to_numpy())


# Animation settings
transition_frames = 30
hold_frames = 20

fig, ax = plt.subplots(figsize=(8, 6))

ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

ax.set_xlabel("x")
ax.set_ylabel("y")

scatter = ax.scatter([], [], s=25)

# Regression line
regression_line, = ax.plot([], [], lw=2)

title = ax.set_title("")

# Statistics text
stats_text = ax.text(
    0.03, 0.97,
    "",
    transform=ax.transAxes,
    va='top',
    fontsize=10,
    family='monospace'
)


def update(frame):

    block_size = transition_frames + hold_frames

    dataset_index = (frame // block_size) % len(datasets)
    frame_in_block = frame % block_size

    current = points[dataset_index]
    next_points = points[(dataset_index + 1) % len(datasets)]

    # Hold current dataset
    if frame_in_block < hold_frames:

        xy = current
        title.set_text(datasets[dataset_index])

    # Transition to next dataset
    else:

        t = (frame_in_block - hold_frames) / transition_frames

        # Smooth interpolation
        t = 3*t**2 - 2*t**3

        xy = (1 - t) * current + t * next_points

        title.set_text(
            f"{datasets[dataset_index]} → "
            f"{datasets[(dataset_index + 1) % len(datasets)]}"
        )

    # Update scatter points
    scatter.set_offsets(xy)

    x = xy[:, 0]
    y = xy[:, 1]

    # --------------------------------------------------
    # Statistics
    # --------------------------------------------------

    mean_x = np.mean(x)
    mean_y = np.mean(y)

    # Sample variance: denominator = n - 1
    var_x = np.var(x, ddof=1)
    var_y = np.var(y, ddof=1)

    correlation = np.corrcoef(x, y)[0, 1]

    # Linear regression
    slope, intercept = np.polyfit(x, y, 1)

    predicted_y = slope * x + intercept

    # R-squared
    ss_res = np.sum((y - predicted_y)**2)
    ss_tot = np.sum((y - mean_y)**2)

    r_squared = 1 - ss_res / ss_tot

    # --------------------------------------------------
    # Regression line
    # --------------------------------------------------

    x_line = np.array([0, 100])
    y_line = slope * x_line + intercept

    regression_line.set_data(x_line, y_line)

    # --------------------------------------------------
    # Update statistics text
    # --------------------------------------------------

    stats_text.set_text(
        f"mean(x) = {mean_x:6.2f}\n"
        f"mean(y) = {mean_y:6.2f}\n"
        f"var(x)  = {var_x:6.2f}\n"
        f"var(y)  = {var_y:6.2f}\n"
        f"r       = {correlation:6.3f}\n"
        f"y = {slope:.3f}x + {intercept:.3f}\n"
        f"R²      = {r_squared:.3f}"
    )

    return scatter, regression_line, title, stats_text


frames = len(datasets) * (transition_frames + hold_frames)

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=40,
    blit=False
)

plt.close()

HTML(ani.to_jshtml())